In [53]:
import heapq
import numpy as np
import pandas as pd
import plotly.express as px
import collections
import itertools
import tqdm
from copy import deepcopy
from decimal import *

In [54]:
getcontext().prec = 50

In [30]:
np.random.seed(0)

In [31]:
def normalize_priors(priors):
    if np.sum(priors) == 0:
        return np.zeros_like(priors)
    return priors / np.sum(priors)

In [71]:
def best_response(x, thresholds, priors, c):
    posteriors = normalize_priors(priors)
    search_space = [x] + thresholds[thresholds > x].tolist()
    utilities = []
    for i, x_p in enumerate(search_space):
        utility = np.dot(posteriors, x_p >= thresholds)
        cost = c * abs(x-x_p)
        utilities.append(utility - cost + (i*1e-6))
    return search_space[np.argmax(utilities)]

def best_response_vectorized(X, thresholds, priors, c):
    posteriors = normalize_priors(priors).flatten()
    utilities_expected = np.array([
        np.dot(posteriors, thresholds[j] >= thresholds)
        for j in range(len(thresholds))
    ])

    pass_matrix = X[:, None] >= thresholds[None, :]
    utility_stay = pass_matrix @ posteriors

    X_col = X[:, None]
    thresholds_row = thresholds[None, :]

    feasible = thresholds_row > X_col
    utility_jump = utilities_expected[None, :] - c * np.abs(thresholds_row - X_col)
    utility_jump[~feasible] = -np.inf

    utilities = np.concatenate([utility_stay[:, None], utility_jump],axis=1)

    best_idx = np.argmax(utilities + np.array([(utilities.shape[1] - i)*1e-6 for i in range(utilities.shape[1])]), axis=1)
    X_p = np.where(best_idx == 0, X, thresholds[best_idx - 1])

    return X_p

In [33]:
def accuracy_loss_vectorized(X, X_p, thresholds, priors, threshold_true):
    Y_true = (X >= threshold_true).astype(float)
    Y_p = (X_p[:, None] >= thresholds[None, :]).astype(float)
    losses = np.abs(Y_true[:, None] - Y_p).mean(axis=0)
    posteriors = normalize_priors(priors)
    return np.dot(losses, posteriors)

In [34]:
def evaluate_partition(X, partition, thresholds, priors, threshold_true, c):
    thresholds_p = thresholds[partition]
    priors_p     = priors[partition]
    X_p = best_response_vectorized(X, thresholds_p, priors_p, c)
    return accuracy_loss_vectorized(X, X_p, thresholds_p, priors_p, threshold_true)

def evaluate_system(X, partitions, thresholds, priors, threshold_true, c):
    acc_loss = 0.0
    for partition in partitions:
        acc_loss_p  = evaluate_partition(X, partition, thresholds, priors, threshold_true, c)
        acc_loss += acc_loss_p * np.sum(priors[partition])
    return acc_loss

In [35]:
def set_partitions(collection):
    if len(collection) == 1:
        yield [collection]
        return

    first = collection[0]
    for smaller in set_partitions(collection[1:]):
        for i in range(len(smaller)):
            yield smaller[:i] + [[first] + smaller[i]] + smaller[i+1:]
        yield [[first]] + smaller

In [36]:
def find_partitions_optimal(X, thresholds, priors, threshold_true, c):
    indices = list(range(len(thresholds)))
    partitions_set = list(set_partitions(indices))
    best_partition, best_loss = None, np.inf
    for partitions in partitions_set:
        acc_loss = evaluate_system(X, partitions, thresholds, priors, threshold_true, c)
        if acc_loss < best_loss:
            best_loss      = acc_loss
            best_partition = partitions
    return best_partition

In [37]:
def display_priority_queue(pq, P):
    res = "[  "
    for acc_loss, (a_id, b_id) in pq:
        res += f"({-acc_loss:.4e}, ({P[a_id]}, {P[b_id]}))  "
    res += "]"
    print(res)

def find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=False):
    P = {}
    next_id = 0
    partitions = [[i] for i in range(len(priors))]
    for block in partitions:
        P[next_id] = list(block)
        next_id += 1
    
    pq = []
    for a_id, b_id in itertools.combinations(P.keys(), 2):
        a, b = P[a_id], P[b_id]
        ab = sorted(a+b)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c) * np.sum(priors[a])
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c) * np.sum(priors[b])
        gain = -(acc_loss_a + acc_loss_b - acc_loss_ab)
        heapq.heappush(pq, (gain, (a_id, b_id)))

    while pq:
        if display:
            display_priority_queue(pq, P)
        gain, (a_id, b_id) = heapq.heappop(pq)
        if a_id not in P.keys() or b_id not in P.keys():
            continue

        a = P[a_id]
        b = P[b_id]

        ab = sorted(a + b)
        acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
        acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)
        acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
        lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
        rhs = acc_loss_ab * np.sum(priors[ab])
        # print(ab,  lhs, rhs)

        if lhs - rhs > -1e-9:
            del P[a_id]
            del P[b_id]

            pq2 = []
            for acc_loss, (x_id, y_id) in pq:
                if x_id not in {a_id, b_id} and y_id not in {a_id, b_id}:
                    heapq.heappush(pq2, (acc_loss, (x_id, y_id)))
            pq = deepcopy(pq2)

            new_id = next_id
            next_id += 1
            P[new_id] = ab

            for p_id in P.keys():
                if p_id != new_id:
                    p = P[p_id]
                    merged = sorted(ab + p)
                    acc_loss_merged = evaluate_partition(X, merged, thresholds, priors, threshold_true, c) * np.sum(priors[merged])
                    acc_loss_p = evaluate_partition(X, p, thresholds, priors, threshold_true, c) * np.sum(priors[p])
                    acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c) * np.sum(priors[ab])
                    gain = -(acc_loss_p + acc_loss_ab - acc_loss_merged)
                    heapq.heappush(pq, (gain, (new_id, p_id)))
    return list(P.values())

In [38]:
def approximation_ratio(loss_optimal, loss_greedy, rtype="m"):
    if rtype in ["a", "add", "additive"]:
        return loss_greedy - loss_optimal
    elif rtype in ["m", "mult", "multiplicative"]:
        if loss_optimal == 0:
            return np.nan
        return loss_greedy / loss_optimal

In [39]:
x_min, x_max, x_disc = 0., 1., 1e-4
X = np.arange(x_min, x_max+x_disc, x_disc).round(4)

In [40]:
df = pd.read_pickle("../results/grid_search_approx_ratio_best_n4_fix.pkl")

In [41]:
df.head()

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors
0,0.1,0.75,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.09999,0.09999,1.0,0.0,"[0.0, 0.2, 0.4, 0.6]","[1.0, 0.0, 0.0, 0.0]"
1,0.2,0.75,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.19998,0.19998,1.0,0.0,"[0.0, 0.2, 0.4, 0.6]","[1.0, 0.0, 0.0, 0.0]"
2,0.3,0.75,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.29997,0.29997,1.0,0.0,"[0.0, 0.2, 0.4, 0.6]","[1.0, 0.0, 0.0, 0.0]"
3,0.4,0.75,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.39996,0.39996,1.0,0.0,"[0.0, 0.2, 0.4, 0.6]","[1.0, 0.0, 0.0, 0.0]"
4,0.5,0.75,"[[0, 1, 2, 3]]","[[0, 1, 2, 3]]",0.49995,0.49995,1.0,0.0,"[0.0, 0.2, 0.4, 0.6]","[1.0, 0.0, 0.0, 0.0]"


In [42]:
df[df["r_mult"]>2.5]

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors
551400,0.1,0.75,"[[1], [0, 2, 3]]","[[0], [1, 2, 3]]",0.038926,0.09999,2.568713,0.061064,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.08, 0.18, 0.44]"
1254030,0.1,0.75,"[[1], [0, 2, 3]]","[[3], [0, 1, 2]]",0.038926,0.09999,2.568713,0.061064,"[0.0, 0.2, 0.8, 1.0]","[0.3, 0.08, 0.48, 0.14]"
2191070,0.1,0.75,"[[1], [0, 2, 3]]","[[3], [0, 1, 2]]",0.038926,0.09999,2.568713,0.061064,"[0.0, 0.6, 0.8, 1.0]","[0.3, 0.08, 0.48, 0.14]"


In [43]:
df["opt_len"] = df["partition_opt"].apply(lambda x: len(x))
df["greedy_len"] = df["partition_greedy"].apply(lambda x: len(x))

In [44]:
df_bad = df
df_bad = df_bad.sort_values("r_mult", ascending=False).reset_index(drop=True)
df_bad.head()

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors,opt_len,greedy_len
0,0.1,0.75,"[[1], [0, 2, 3]]","[[0], [1, 2, 3]]",0.038926,0.099990,2.568713,0.061064,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.08, 0.18, 0.44]",2,2
1,0.1,0.75,"[[1], [0, 2, 3]]","[[3], [0, 1, 2]]",0.038926,0.099990,2.568713,0.061064,"[0.0, 0.2, 0.8, 1.0]","[0.3, 0.08, 0.48, 0.14]",2,2
2,0.1,0.75,"[[1], [0, 2, 3]]","[[3], [0, 1, 2]]",0.038926,0.099990,2.568713,0.061064,"[0.0, 0.6, 0.8, 1.0]","[0.3, 0.08, 0.48, 0.14]",2,2
3,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.040988,0.099990,2.439500,0.059002,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.06, 0.2, 0.44]",2,2
4,0.1,0.75,"[[1], [0, 2, 3]]","[[0, 1], [2, 3]]",0.038926,0.093535,2.402877,0.054609,"[0.0, 0.4, 0.6, 1.0]","[0.3, 0.08, 0.34, 0.28]",2,2


In [129]:
i = df_bad["r_mult"].argmax()
# i = 1
df_bad.iloc[[i]]

,threshold_true,c,partition_opt,partition_greedy,acc_loss_opt,acc_loss_greedy,r_mult,r_add,thresholds,priors,opt_len,greedy_len
0,0.1,0.75,"[[1], [0, 2, 3]]","[[0], [1, 2, 3]]",0.038926,0.09999,2.568713,0.061064,"[0.0, 0.2, 0.4, 1.0]","[0.3, 0.08, 0.18, 0.44]",2,2


In [130]:
thresholds = df_bad["thresholds"].iloc[i]
priors = df_bad["priors"].iloc[i]
threshold_true = df_bad["threshold_true"].iloc[i]
c = df_bad["c"].iloc[i]
partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c, display=True)
loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
r = approximation_ratio(loss_opt, loss_greedy)

[  (6.9389e-18, ([1], [3]))  (3.4694e-18, ([1], [2]))  (-6.9389e-18, ([2], [3]))  (-6.9389e-18, ([0], [2]))  (-6.9389e-18, ([0], [1]))  (-3.2117e-03, ([0], [3]))  ]
[  (0.0000e+00, ([1, 3], [2]))  (-3.1621e-02, ([1, 3], [0]))  (-6.9389e-18, ([0], [2]))  ]
[  (-8.0004e-02, ([1, 2, 3], [0]))  ]


In [127]:
display(pd.DataFrame({"Thresholds": thresholds, "Priors": priors}).round(6).T)
print(f"c              : {c}")
print(f"t*             : {threshold_true:.4f}")
print()
print("Greedy")
print("------")
print(f"Partition      : {partition_greedy}")
print(f"Accuracy loss  : {loss_greedy:.6f}")
print()
print("Optimal")
print("-------")
print(f"Partition      : {partition_opt}")
print(f"Accuracy loss  : {loss_opt:.6f}")
print()
print(f"r (mult)       : {r}")
print("-"*80)
print()

,0,1,2,3
Thresholds,0.0,0.20,0.40,1.00
Priors,0.3,0.08,0.18,0.44


c              : 0.75
t*             : 0.1000

Greedy
------
Partition      : [[0], [1, 2, 3]]
Accuracy loss  : 0.099990

Optimal
-------
Partition      : [[1], [0, 2, 3]]
Accuracy loss  : 0.038926

r (mult)       : 2.568713074749551
--------------------------------------------------------------------------------



In [117]:
a, b = [1,3], [2]

acc_loss_a = evaluate_partition(X, a, thresholds, priors, threshold_true, c)
acc_loss_b = evaluate_partition(X, b, thresholds, priors, threshold_true, c)

lhs = acc_loss_a * np.sum(priors[a]) + acc_loss_b * np.sum(priors[b])
ab = sorted(a + b)

acc_loss_ab = evaluate_partition(X, ab, thresholds, priors, threshold_true, c)
rhs = acc_loss_ab * np.sum(priors[ab])
gain = lhs - rhs

print(f"    threshold true: {threshold_true:.4f}")
print(f"                 c: {c}")
print(f"                 a: {a}")
print(f"                 b: {b}")
print(f"   accuracy loss a: {acc_loss_a:.7f}")
print(f"   accuracy loss b: {acc_loss_b:.7f}")
print(f"  accuracy loss ab: {acc_loss_ab:.7f}")
print(f"               LHS: {lhs:.7f}")
print(f"               RHS: {rhs:.7f}")
print(f"              gain: {gain}")
print(f"            merge?: {lhs - rhs > -1e-6}")
print()

    threshold true: 0.1000
                 c: 0.75
                 a: [1, 3]
                 b: [2]
   accuracy loss a: 0.0999900
   accuracy loss b: 0.0999900
  accuracy loss ab: 0.0999900
               LHS: 0.0699930
               RHS: 0.0699930
              gain: 0.0
            merge?: True



In [122]:
p = [1, 2]
X_p = best_response_vectorized(X, thresholds[p], priors[p], c)
px.scatter(x=X, y=X_p)

In [28]:
# threshold_trues = np.arange(0.09, 0.11, 1e-5)
# approx_ratios = []
# for threshold_true in tqdm.tqdm(threshold_trues):
#     partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
#     partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
#     loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
#     loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
#     r = approximation_ratio(loss_opt, loss_greedy)
#     approx_ratios.append(r)

# print()
# px.scatter(y=approx_ratios, x=threshold_trues)

In [ ]:
epsilons = np.arange(0, 0.08+1e-4, 1e-4).round(4)
approx_ratios = []
results_eps = {"epsilon": [], "threshold": [], "priors": [], "partition_greedy": [], "partition_opt": [], "r": [], "error": []}
ec = 0
for eps in tqdm.tqdm(epsilons):
    priors =  np.array([0.3/(1-eps), 0.08-eps, 0.18/(1-eps), 0.44/(1-eps)])
    remainder = 1 - np.sum(priors)
    priors[0] += remainder
    if np.sum(priors) != 1:
        ec += 1
        error = 1
    else:
        error = 0
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_eps["epsilon"].append(eps)
    results_eps["threshold"].append(thresholds)
    results_eps["priors"].append(priors)
    results_eps["partition_greedy"].append(f"{partition_greedy}")
    results_eps["partition_opt"].append(f"{partition_opt}")
    results_eps["r"].append(r.item())
    results_eps["error"].append(error)

df_eps = pd.DataFrame(results_eps)

100%|██████████| 801/801 [00:26<00:00, 30.62it/s]


In [103]:
epsilons = np.arange(0, 801)  # integer steps instead of floats
approx_ratios = []
results_eps = {"epsilon": [], "threshold": [], "priors": [], "partition_greedy": [], "partition_opt": [], "r": [], "error": []}
ec = 0

for eps_int in tqdm.tqdm(epsilons):
    eps = eps_int * 1e-4                        # exact float: 0, 0.0001, 0.0002...
    one_minus_eps = (10000 - eps_int) / 10000   # exact: integer division, no accumulation

    priors = np.array([0.3/one_minus_eps, 0.08-eps, 0.18/one_minus_eps, 0.44/one_minus_eps])
    remainder = 1 - np.sum(priors)
    priors[0] += remainder
    if np.sum(priors) != 1:
        ec += 1
        error = 1
    else:
        error = 0
    partition_opt = find_partitions_optimal(X, thresholds, priors, threshold_true, c)
    partition_greedy = find_partitions_greedy_best(X, thresholds, priors, threshold_true, c)
    loss_opt = evaluate_system(X, partition_opt, thresholds, priors, threshold_true, c)
    loss_greedy = evaluate_system(X, partition_greedy, thresholds, priors, threshold_true, c)
    r = approximation_ratio(loss_opt, loss_greedy)
    
    results_eps["epsilon"].append(eps)
    results_eps["threshold"].append(thresholds)
    results_eps["priors"].append(priors)
    results_eps["partition_greedy"].append(f"{partition_greedy}")
    results_eps["partition_opt"].append(f"{partition_opt}")
    results_eps["r"].append(r.item())
    results_eps["error"].append(error)

df_eps = pd.DataFrame(results_eps)

100%|██████████| 801/801 [00:26<00:00, 30.70it/s]


In [105]:
ec

32

In [107]:
px.line(df_eps, x="epsilon", y="r", hover_data=["partition_greedy", "partition_opt"])